# Inter-coder reliability: topics

## Setup

In [ ]:
import pandas as pd
import csv
import numpy as np
# get working directory
import os
import simpledorff
os.getcwd()
# set working directory
os.chdir('analyses/annotation/reliability')

In [ ]:
# Import csv file
df = pd.read_csv('reliability_topics_final_extra.csv', delimiter = ';',  quoting=csv.QUOTE_NONNUMERIC, encoding='utf-8')

# make article_id integer
df['article_id'] = df['article_id'].astype(int)

print(df.shape)

In [ ]:
# any duplicates based on article_id and coder? 
df.duplicated(subset=['article_id', 'coder']).sum()

duplicated = df[df.duplicated(subset=['article_id', 'coder'], keep=False)] 

In [ ]:
for i in df.columns:
    print(i)

In [ ]:
print(df.shape)

In [ ]:
for i in df.columns:
    if i.startswith('topic_'):

In [ ]:
def change_to_binary(x):
    # if x is NaN then 0 else 1
    if pd.isnull(x) | (x == 0):
        return 0
    else:
        return 1

# Assuming df is your DataFrame
topic_columns = df.filter(like='topic_')
df[topic_columns.columns] = topic_columns.applymap(change_to_binary)

In [ ]:
for i in df.columns:
    if i.startswith('topic_'):

In [ ]:
# df['about_covid'] = df['about_covid'].apply(change_to_binary)

# make about_covid integer
df['about_covid'] = df['about_covid'].astype(int)

# make topic columns integer
for i in df.columns:
    if i.startswith('topic_'):
        df[i] = df[i].astype(int)

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(df,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col="about_covid"))

In [ ]:
df_covid = df[df['about_covid']==1]
# count number of coders per article_id
nr_coders = df_covid.groupby('article_id')['coder'].count().reset_index()
articles_to_remove = nr_coders[nr_coders['coder']<2]['article_id']

df_covid = df_covid[~df_covid['article_id'].isin(articles_to_remove)]

nr_coders = df_covid.groupby('article_id')['coder'].count().reset_index()

In [ ]:
for i in df_covid.columns:
    if i.startswith('topic_') & (i != 'topic_o_text') & (i != 'topic_o'):
        print(i)
        print(simpledorff.calculate_krippendorffs_alpha_for_df(df_covid,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col=i))

In [ ]:
# create topic_ij if topic_i or topic_j is 1 then topic_ij is 1 else 0
df_covid['topic_ij'] = np.where((df_covid['topic_i'] == 1) | (df_covid['topic_j'] == 1), 1, 0)
pd.crosstab(df_covid['topic_i'], df_covid['topic_j'])

In [ ]:
# calculate alpha for topic_ij
simpledorff.calculate_krippendorffs_alpha_for_df(df_covid,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col='topic_ij')

# Get articles coded different by two coders and correct where they disagree

In [ ]:
df.to_csv('correction_researcher/reliability_topics_final_extra_binary.csv', index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC, encoding='utf-8')

In [ ]:
# read in corrected df 

corrected_df = pd.read_csv('correction_researcher/reliability_topics_researcher.csv', delimiter = ';',  quoting=csv.QUOTE_NONNUMERIC, encoding='utf-8')
corrected_df['article_id'] = corrected_df['article_id'].astype(int)

# apply change to binary to all columns starting from about_covid
binary_cols = ['about_covid', 'topic_a',
       'topic_b', 'topic_c', 'topic_d', 'topic_e', 'topic_f', 'topic_g',
       'topic_h', 'topic_i', 'topic_j', 'topic_k', 'topic_l', 'topic_m',
       'topic_n', 'other_country_binary', 'actors_present']

for i in binary_cols:
    corrected_df[i] = corrected_df[i].apply(change_to_binary)

print(corrected_df.shape)

In [ ]:
# calculate the number of unique coder per article_id
nr_coders = corrected_df.groupby('article_id')['coder'].count().reset_index()

In [ ]:
# filter data to be between researcher and second_coder
df_researcher_second_coder = corrected_df[corrected_df['coder'].isin(['researcher', 'second_coder'])]
df_researcher_main_coder = corrected_df[corrected_df['coder'].isin(['researcher', 'main_coder'])]
df_main_coder_second_coder = corrected_df[corrected_df['coder'].isin(['main_coder', 'second_coder'])]

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(corrected_df,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col="about_covid"))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(df_researcher_second_coder,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col="about_covid"))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(df_researcher_main_coder,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col="about_covid"))

In [ ]:
print(simpledorff.calculate_krippendorffs_alpha_for_df(df_main_coder_second_coder,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col="about_covid"))

In [ ]:
df_covid_new = corrected_df[corrected_df['about_covid']==1]
# count number of coders per article_id
nr_coders = df_covid_new.groupby('article_id')['coder'].count().reset_index()
articles_to_remove = nr_coders[nr_coders['coder']<3]['article_id']

In [ ]:
df_covid_new = df_covid_new[~df_covid_new['article_id'].isin(articles_to_remove)]

nr_coders = df_covid_new.groupby('article_id')['coder'].count().reset_index()

In [ ]:
# filter data to be between researcher and second_coder
df_researcher_second_coder_topics = df_covid_new[df_covid_new['coder'].isin(['researcher', 'second_coder'])]
df_researcher_main_coder_topics = df_covid_new[df_covid_new['coder'].isin(['researcher', 'main_coder'])]
df_main_coder_second_coder_topics = df_covid_new[df_covid_new['coder'].isin(['main_coder', 'second_coder'])]

In [ ]:
for i in df_covid_new.columns:
    if i.startswith('topic_') & (i != 'topic_o_text') & (i != 'topic_o'):
        print(i)
        print(simpledorff.calculate_krippendorffs_alpha_for_df(df_covid_new,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col=i))

In [ ]:
for i in df_researcher_second_coder_topics.columns:
    if i.startswith('topic_') & (i != 'topic_o_text') & (i != 'topic_o'):
        print(i)
        print(simpledorff.calculate_krippendorffs_alpha_for_df(df_researcher_second_coder_topics,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col=i))

In [ ]:
for i in df_researcher_main_coder_topics.columns:
    if i.startswith('topic_') & (i != 'topic_o_text') & (i != 'topic_o'):
        print(i)
        print(simpledorff.calculate_krippendorffs_alpha_for_df(df_researcher_main_coder_topics,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col=i))

In [ ]:
for i in df_main_coder_second_coder_topics.columns:
    if i.startswith('topic_') & (i != 'topic_o_text') & (i != 'topic_o'):
        print(i)
        print(simpledorff.calculate_krippendorffs_alpha_for_df(df_main_coder_second_coder_topics,experiment_col='article_id',
                                                    annotator_col='coder',
                                                    class_col=i))